# Scenario-count sensitivity: Financial Chronos (46M)

Does drawing more sampled trajectories change the M6 forecast? This notebook
repeats the same zero-shot inference as the main Financial Chronos run and
varies only the number of trajectories drawn per asset.

| | |
|---|---|
| Model | `FinText/Chronos_Small_2021_Global`, zero-shot |
| Variable | `num_samples` = 300, 500 and 1000 (the main run used 100) |
| Fixed | context 512 weekdays, horizon 20, seed 42, official M6 asset order |
| Input | the 12 shared round-context CSVs on Drive |
| Output | 36 NPZ files, copied afterwards into the repository |

Asking for 300 or more samples in one `predict()` call exhausts the GPU, so each
round draws 1000 trajectories as ten consecutive 100-sample chunks and the
smaller files are prefixes of that sequence. The seed is set once per round and
never reset between chunks, so the chunks differ and the first n columns are
always a valid n-sample draw.

Because the three counts are nested this way, a difference between them reflects
the extra samples rather than a different random draw. None is nested inside the
main 100-sample run, which was a separate draw, so part of any gap against 100
is Monte Carlo noise.

Each round is written to Drive and verified before the next begins, so a
disconnect costs only the round in progress.

## 1. Install Chronos

Colab may ask you to restart the runtime. If it does, restart and re-run from
this cell.

In [ ]:
%pip install -q chronos-forecasting

## 2. Imports and settings

`SAMPLE_COUNTS` and the chunking constants are the only settings that differ
from the main Financial Chronos notebook. `SAMPLE_CHUNK_SIZE = 100` is the request
size already known to fit this GPU at `SERIES_BATCH_SIZE = 10`, and every round
draws `MAX_SAMPLES` in that many chunks. `ROUND_SCHEDULE` and the expected
leading-NaN pattern are carried over unchanged, so a wrong context file is
caught before inference.

In [ ]:
import traceback
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from chronos import ChronosPipeline

# --- Model (unchanged) ------------------------------------------------------
MODEL_ID = "FinText/Chronos_Small_2021_Global"

# --- The experimental variable ---------------------------------------------
SAMPLE_COUNTS = [300, 500, 1000]      # main run used 100

# Requesting 300+ samples in one predict() call exhausts the L4's memory, so each
# round draws MAX_SAMPLES in chunks of SAMPLE_CHUNK_SIZE - the request size
# already known to work at SERIES_BATCH_SIZE = 10 - and the configurations above
# are taken as prefixes of that single sampled sequence.
SAMPLE_CHUNK_SIZE = 100
MAX_SAMPLES = max(SAMPLE_COUNTS)
N_CHUNKS, remainder = divmod(MAX_SAMPLES, SAMPLE_CHUNK_SIZE)
assert remainder == 0, "MAX_SAMPLES must be a whole number of chunks"
assert all(n % SAMPLE_CHUNK_SIZE == 0 for n in SAMPLE_COUNTS), (
    "every sample count must be a whole number of chunks"
)

# --- Everything else is fixed, exactly as in the main run -------------------
CONTEXT_LENGTH = 512
PREDICTION_LENGTH = 20
RANDOM_SEED = 42
SERIES_BATCH_SIZE = 10                # lower ONLY on CUDA OOM
ROUNDS = range(1, 13)


# --- Official M6 round schedule (context-building step, pre-specified) --------------------
# round -> context start, origin (= context end), forecast start, forecast end
ROUND_SCHEDULE = {
    1:  ("2020-03-19", "2022-03-04", "2022-03-07", "2022-04-01"),
    2:  ("2020-04-16", "2022-04-01", "2022-04-04", "2022-04-29"),
    3:  ("2020-05-14", "2022-04-29", "2022-05-02", "2022-05-27"),
    4:  ("2020-06-11", "2022-05-27", "2022-05-30", "2022-06-24"),
    5:  ("2020-07-09", "2022-06-24", "2022-06-27", "2022-07-22"),
    6:  ("2020-08-06", "2022-07-22", "2022-07-25", "2022-08-19"),
    7:  ("2020-09-03", "2022-08-19", "2022-08-22", "2022-09-16"),
    8:  ("2020-10-01", "2022-09-16", "2022-09-19", "2022-10-14"),
    9:  ("2020-10-29", "2022-10-14", "2022-10-17", "2022-11-11"),
    10: ("2020-11-26", "2022-11-11", "2022-11-14", "2022-12-09"),
    11: ("2020-12-24", "2022-12-09", "2022-12-12", "2023-01-06"),
    12: ("2021-01-21", "2023-01-06", "2023-01-09", "2023-02-03"),
}

# Genuine leading missing history, per context-building step (verified against the repo files).
EXPECTED_LEADING_NAN = {r: {"OGN": 302 - 20 * (r - 1)} for r in ROUNDS}
EXPECTED_LEADING_NAN[1] = {"CARR": 1, "OGN": 302}

# Official M6 asset order - used to verify the contexts, not to reorder them.
OFFICIAL_ASSET_ORDER = [
    "ABBV", "ACN", "AEP", "AIZ", "ALLE", "AMAT", "AMP", "AMZN", "AVB", "AVY",
    "AXP", "BDX", "BF-B", "BMY", "BR", "CARR", "CDW", "CE", "CHTR", "CNC",
    "CNP", "COP", "CTAS", "CZR", "DG", "DPZ", "DRE", "DXC", "EWA", "EWC",
    "EWG", "EWH", "EWJ", "EWL", "EWQ", "EWT", "EWU", "EWY", "EWZ", "FTV",
    "GOOG", "GPC", "GSG", "HIG", "HIGH.L", "HST", "HYG", "IAU", "ICLN",
    "IEAA.L", "IEF", "IEFM.L", "IEMG", "IEUS", "IEVL.L", "IGF", "INDA",
    "IUMO.L", "IUVL.L", "IVV", "IWM", "IXN", "JPEA.L", "JPM", "KR", "LQD",
    "MCHI", "META", "MVEU.L", "OGN", "PG", "PPL", "PRU", "PYPL", "RE",
    "REET", "ROL", "ROST", "SEGA.L", "SHY", "SLV", "SPMV.L", "TLT", "UNH",
    "URI", "V", "VRSK", "VXX", "WRK", "XLB", "XLC", "XLE", "XLF", "XLI",
    "XLK", "XLP", "XLU", "XLV", "XLY", "XOM",
]
N_ASSETS = len(OFFICIAL_ASSET_ORDER)

TOTAL_RUNS = len(SAMPLE_COUNTS) * len(list(ROUNDS))

print(f"chronos-forecasting {version('chronos-forecasting')} | torch {torch.__version__}")
print(f"Experiment: {MODEL_ID}")
print(f"Sample counts: {SAMPLE_COUNTS} x rounds {min(ROUNDS)}-{max(ROUNDS)} "
      f"= {TOTAL_RUNS} saved files")
print(f"Generation: {MAX_SAMPLES} samples per round in {N_CHUNKS} chunks of "
      f"{SAMPLE_CHUNK_SIZE}, seed {RANDOM_SEED} set once per round")
for n in SAMPLE_COUNTS:
    print(f"  {n:>4} samples = first {n // SAMPLE_CHUNK_SIZE} chunk(s) -> "
          f"({N_ASSETS}, {n}, {PREDICTION_LENGTH})")

## 3. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Configure paths

Contexts are read from the shared folder that every run in this project uses.
`DRIVE_OUTPUT_BASE_DIR` is the only path specific to this experiment; the
`samples_300/`, `samples_500/` and `samples_1000/` subfolders are created automatically, so
no configuration can overwrite another.

In [ ]:
# Existing shared context folder - same files as every previous run.
DRIVE_CONTEXT_DIR = Path("/content/drive/MyDrive/HonoursResearch/contexts")

# >>> THE ONLY NEW PATH TO CONFIGURE <<<
DRIVE_OUTPUT_BASE_DIR = Path("/content/drive/MyDrive/HonoursResearch/outputs/financial_chronos")


def context_path(round_number: int) -> Path:
    return DRIVE_CONTEXT_DIR / f"round_{round_number:02d}_context.csv"


def config_dir(num_samples: int) -> Path:
    return DRIVE_OUTPUT_BASE_DIR / f"samples_{num_samples}"


def samples_path(round_number: int, num_samples: int) -> Path:
    return (config_dir(num_samples)
            / f"financial_chronos_round{round_number:02d}_samples{num_samples}.npz")


if not DRIVE_CONTEXT_DIR.is_dir():
    raise FileNotFoundError(
        f"Context folder not found:\n  {DRIVE_CONTEXT_DIR}\n"
        "It must be the SAME folder used by the main Financial Chronos notebook, "
        "containing round_01_context.csv .. round_12_context.csv."
    )

for n in SAMPLE_COUNTS:
    config_dir(n).mkdir(parents=True, exist_ok=True)

print(f"Contexts (existing, shared): {DRIVE_CONTEXT_DIR}")
print(f"Output base (new)          : {DRIVE_OUTPUT_BASE_DIR}")
for n in SAMPLE_COUNTS:
    print(f"  {config_dir(n)}")
print(f"\nExample filename: {samples_path(1, SAMPLE_COUNTS[0]).name}")

## 5. Check the 12 context files are present

In [ ]:
missing = [context_path(r).name for r in ROUNDS if not context_path(r).is_file()]
if missing:
    raise FileNotFoundError(
        f"Missing context files in {DRIVE_CONTEXT_DIR}:\n  " + "\n  ".join(missing))

print(f"All {len(list(ROUNDS))} context files present in {DRIVE_CONTEXT_DIR}")

## 6. GPU check

A GPU is required. This cell stops rather than silently running on CPU.

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU detected. Runtime > Change runtime type > Hardware "
        "accelerator: GPU, then re-run from section 3."
    )

DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
GPU_NAME = torch.cuda.get_device_name(0)

print(f"GPU   : {GPU_NAME}")
print(f"dtype : {DTYPE}")

## 7. Load the model, once

Same checkpoint, CUDA and bfloat16 setup as the main Financial Chronos run. Loaded
here and reused for every round, never reloaded inside the loops. Inference
only: no training, fine-tuning or weight modification.

In [ ]:
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed_all(RANDOM_SEED)

pipeline = ChronosPipeline.from_pretrained(
    MODEL_ID,
    device_map="cuda",
    torch_dtype=DTYPE,
)

n_params = sum(p.numel() for p in pipeline.model.parameters())
MODEL_LOADED_ONCE = True

print(f"Loaded: {MODEL_ID}")
print(f"  parameters : {n_params:,} ({n_params / 1e6:.1f}M)")
print(f"  device     : {GPU_NAME} | dtype {DTYPE}")
print("Model loaded ONCE - reused for every sample count and every round.")

## 8. Helpers

Three steps per round. `load_and_validate_context` reads one context file and
checks its shape, dates, asset order and missing-value pattern.
`generate_round_samples` draws `MAX_SAMPLES` trajectories in chunks, releasing
each GPU tensor as it goes; the seed is set once at the start of the round, and
an assertion confirms chunk 2 differs from chunk 1, since matching chunks would
mean the seed had been reset. `save_and_verify_round` writes one sample count
and reads it back to confirm it is intact.

In [ ]:
def load_and_validate_context(round_number: int):
    """Load one model context, validate it, and build the model input."""
    ctx_start, origin, fc_start, fc_end = ROUND_SCHEDULE[round_number]
    df = pd.read_csv(context_path(round_number), parse_dates=["date"])

    assert df.shape[0] == CONTEXT_LENGTH, (
        f"Round {round_number}: expected {CONTEXT_LENGTH} rows, found {df.shape[0]}"
    )
    assert list(df.columns) == ["date"] + OFFICIAL_ASSET_ORDER, (
        f"Round {round_number}: columns are not 'date' + the official M6 asset order"
    )
    dates = df["date"]
    assert dates.is_monotonic_increasing, f"Round {round_number}: dates not ascending"
    assert not dates.duplicated().any(), f"Round {round_number}: duplicate dates"
    assert dates.iloc[0] == pd.Timestamp(ctx_start), f"Round {round_number}: context start"
    assert dates.iloc[-1] == pd.Timestamp(origin), f"Round {round_number}: context end/origin"
    assert (dates <= pd.Timestamp(origin)).all(), (
        f"Round {round_number}: context contains a date after the origin"
    )

    matrix = df[OFFICIAL_ASSET_ORDER].to_numpy(dtype=np.float64).T
    assert matrix.shape == (N_ASSETS, CONTEXT_LENGTH)
    assert np.array_equal(
        matrix, df[OFFICIAL_ASSET_ORDER].to_numpy().T, equal_nan=True
    ), f"Round {round_number}: model input differs from the loaded context values"

    leading_missing, interior_missing = {}, {}
    for i, symbol in enumerate(OFFICIAL_ASSET_ORDER):
        row = matrix[i]
        n_lead = int(np.argmax(~np.isnan(row))) if np.isnan(row[0]) else 0
        if n_lead:
            leading_missing[symbol] = n_lead
        if np.isnan(row[n_lead:]).any():
            interior_missing[symbol] = int(np.isnan(row[n_lead:]).sum())
    assert not interior_missing, (
        f"Round {round_number}: unexpected interior NaNs {interior_missing}"
    )
    assert leading_missing == EXPECTED_LEADING_NAN[round_number], (
        f"Round {round_number}: leading NaN pattern changed - "
        f"expected {EXPECTED_LEADING_NAN[round_number]}, found {leading_missing}"
    )

    forecast_dates = pd.bdate_range(fc_start, fc_end)
    assert len(forecast_dates) == PREDICTION_LENGTH, f"Round {round_number}: forecast window"
    return matrix, df, forecast_dates


def generate_round_samples(context_matrix, round_number: int):
    """Draw MAX_SAMPLES trajectories for one round; returns (100, MAX_SAMPLES, 20).

    The samples are generated in N_CHUNKS consecutive chunks of
    SAMPLE_CHUNK_SIZE, because a single large predict() call exhausts the GPU.
    The seed is set ONCE here, before the first chunk, and deliberately not
    reset between chunks - re-seeding would make every chunk an identical copy
    of the first. Chunks are concatenated along the sample axis in generation
    order, so the first n columns are always a valid n-sample draw.
    """
    # Seed once per round, so a round can still be rerun reproducibly on its own.
    torch.manual_seed(RANDOM_SEED)
    torch.cuda.manual_seed_all(RANDOM_SEED)

    chunks = []
    for chunk in range(N_CHUNKS):
        batch_outputs = []
        for start in range(0, N_ASSETS, SERIES_BATCH_SIZE):
            stop = min(start + SERIES_BATCH_SIZE, N_ASSETS)
            batch_context = [
                torch.tensor(context_matrix[i], dtype=torch.float32)
                for i in range(start, stop)
            ]
            samples = pipeline.predict(
                inputs=batch_context,
                prediction_length=PREDICTION_LENGTH,
                num_samples=SAMPLE_CHUNK_SIZE,
            )
            batch_outputs.append(samples.to(torch.float32).cpu().numpy())
            # Release the GPU tensor as soon as it is on the host.
            del samples, batch_context
        torch.cuda.empty_cache()

        chunk_samples = np.concatenate(batch_outputs, axis=0)
        assert chunk_samples.shape == (N_ASSETS, SAMPLE_CHUNK_SIZE, PREDICTION_LENGTH), (
            f"Round {round_number} chunk {chunk + 1}: got {chunk_samples.shape}"
        )
        chunks.append(chunk_samples)
        print(f"    chunk {chunk + 1:2d}/{N_CHUNKS} -> {tuple(chunk_samples.shape)} "
              f"(cumulative {(chunk + 1) * SAMPLE_CHUNK_SIZE} samples)")

    forecast_samples = np.concatenate(chunks, axis=1)

    assert forecast_samples.shape == (N_ASSETS, MAX_SAMPLES, PREDICTION_LENGTH), (
        f"Round {round_number}: expected "
        f"({N_ASSETS}, {MAX_SAMPLES}, {PREDICTION_LENGTH}), got {forecast_samples.shape}"
    )
    assert np.isfinite(forecast_samples).all(), (
        f"Round {round_number}: non-finite forecast values"
    )
    # Chunks must differ - identical chunks would mean the seed was reset.
    assert not np.array_equal(
        forecast_samples[:, :SAMPLE_CHUNK_SIZE], forecast_samples[:, SAMPLE_CHUNK_SIZE:2 * SAMPLE_CHUNK_SIZE]
    ), f"Round {round_number}: chunk 2 repeats chunk 1 - the seed was reset mid-round"
    return forecast_samples


def save_and_verify_round(round_number: int, num_samples: int, forecast_samples, forecast_dates):
    """Save the raw array, then reload it and verify shape/order/dates/finiteness."""
    path = samples_path(round_number, num_samples)
    date_strings = np.array([d.strftime("%Y-%m-%d") for d in forecast_dates])

    np.savez_compressed(
        path,
        forecast_samples=forecast_samples,
        asset_symbols=np.array(OFFICIAL_ASSET_ORDER),
        forecast_dates=date_strings,
    )

    with np.load(path, allow_pickle=False) as reloaded:
        r_samples = reloaded["forecast_samples"]
        r_symbols = reloaded["asset_symbols"]
        r_dates = reloaded["forecast_dates"]

    assert r_samples.shape == (N_ASSETS, num_samples, PREDICTION_LENGTH), "reloaded shape"
    assert list(r_symbols) == OFFICIAL_ASSET_ORDER, "asset ordering changed on save/reload"
    assert np.array_equal(r_samples, forecast_samples), "saved values differ"
    assert np.isfinite(r_samples).all(), "reloaded values not finite"
    assert list(r_dates) == list(date_strings), "forecast dates changed on save/reload"
    return path


print("Helpers defined.")

### Run all 12 rounds

Each round draws its 1000 trajectories once, then writes the 300-, 500- and 1000-sample
files from that single sequence before moving on. A failed round is recorded and
the run continues, so one bad round does not abandon the rest.

In [ ]:
records, failures = [], []
round_seconds = {}
experiment_started = datetime.now(timezone.utc)

for r in ROUNDS:
    _, origin, fc_start, fc_end = ROUND_SCHEDULE[r]
    print(f"\n{'=' * 70}\nRound {r:02d} | origin {origin} | forecast {fc_start} .. {fc_end}\n"
          f"{'=' * 70}")
    try:
        context_matrix, context_df, forecast_dates = load_and_validate_context(r)

        started = datetime.now(timezone.utc)
        forecast_samples = generate_round_samples(context_matrix, r)
        elapsed = (datetime.now(timezone.utc) - started).total_seconds()
        round_seconds[r] = elapsed
        print(f"  generated {tuple(forecast_samples.shape)} in {elapsed:.1f}s")

        # Write each configuration as a prefix of the one sampled sequence.
        for num_samples in SAMPLE_COUNTS:
            subset = forecast_samples[:, :num_samples, :].copy()
            saved_path = save_and_verify_round(r, num_samples, subset, forecast_dates)
            records.append({
                "num_samples": num_samples, "round": r, "origin": origin,
                "forecast_start": fc_start, "forecast_end": fc_end,
                "shape": tuple(subset.shape), "file": saved_path.name,
            })
            print(f"    {num_samples:>4} samples -> {tuple(subset.shape)} "
                  f"saved+verified: {saved_path.name}")
            del subset

        del forecast_samples, context_matrix
        torch.cuda.empty_cache()

    except Exception as exc:
        failures.append((r, f"{type(exc).__name__}: {exc}"))
        print(f"  !! ROUND {r:02d} FAILED - completed rounds remain saved")
        traceback.print_exc()
        torch.cuda.empty_cache()

total_minutes = (datetime.now(timezone.utc) - experiment_started).total_seconds() / 60
print(f"\nFinished in {total_minutes:.1f} min. Saved {len(records)}/{TOTAL_RUNS} files "
      f"across {len(round_seconds)}/{len(list(ROUNDS))} completed rounds.")
if failures:
    print(f"FAILED rounds: {[r for r, _ in failures]}")

## 9. Verify every saved file

An independent pass over what is actually on Drive: each NPZ is reopened and
checked for the three expected arrays, the shape for its own sample count, the
official asset order, the scheduled forecast dates and finite values. The last
check confirms each smaller file really is a prefix of the largest one.

In [ ]:
ALL_VERIFIED = True
print(f"{'samples':>7} {'rnd':>4}  {'file':<48} {'status':<7} shape")

for num_samples in SAMPLE_COUNTS:
    for r in ROUNDS:
        path = samples_path(r, num_samples)
        if not path.is_file():
            ALL_VERIFIED = False
            print(f"{num_samples:>7} {r:>4}  {path.name:<48} MISSING")
            continue

        _, _, fc_start, fc_end = ROUND_SCHEDULE[r]
        expected_dates = [d.strftime("%Y-%m-%d") for d in pd.bdate_range(fc_start, fc_end)]
        with np.load(path, allow_pickle=False) as f:
            keys = set(f.files)
            samples = f["forecast_samples"]
            symbols = list(f["asset_symbols"])
            fdates = list(f["forecast_dates"])

        ok = (keys == {"forecast_samples", "asset_symbols", "forecast_dates"}
              and samples.shape == (N_ASSETS, num_samples, PREDICTION_LENGTH)
              and symbols == OFFICIAL_ASSET_ORDER
              and fdates == expected_dates
              and bool(np.isfinite(samples).all()))
        ALL_VERIFIED = ALL_VERIFIED and ok
        print(f"{num_samples:>7} {r:>4}  {path.name:<48} {'OK' if ok else 'FAILED':<7} "
              f"{samples.shape}")

print(f"\nAll {TOTAL_RUNS} saved files verified: {ALL_VERIFIED}")

# The nested design requires each smaller file to be an exact prefix of the
# largest file for the same round.
PREFIX_OK = True
for r in ROUNDS:
    largest = samples_path(r, MAX_SAMPLES)
    if not largest.is_file():
        continue
    with np.load(largest, allow_pickle=False) as f:
        full = f["forecast_samples"]
    for num_samples in SAMPLE_COUNTS:
        if num_samples == MAX_SAMPLES or not samples_path(r, num_samples).is_file():
            continue
        with np.load(samples_path(r, num_samples), allow_pickle=False) as f:
            smaller = f["forecast_samples"]
        if not np.array_equal(smaller, full[:, :num_samples, :]):
            PREFIX_OK = False
            print(f"  round {r:02d}: {num_samples}-sample file is NOT a prefix of the "
                  f"{MAX_SAMPLES}-sample file")

print(f"Nested prefix check (smaller files are prefixes of the "
      f"{MAX_SAMPLES}-sample file): {PREFIX_OK}")

## 10. Completion summary

Shows which configurations finished, so you know what is safe to copy into the
repository for the RPS comparison.

In [ ]:
print("Financial Chronos - sample-count sensitivity test")
print(f"  model        : {MODEL_ID} (loaded once, inference only)")
print(f"  variable     : num_samples in {SAMPLE_COUNTS} (nested prefixes)")
print(f"  generation   : {MAX_SAMPLES} samples per round in {N_CHUNKS} chunks of "
      f"{SAMPLE_CHUNK_SIZE}, seed {RANDOM_SEED} set once per round")
print(f"  fixed        : context {CONTEXT_LENGTH}, horizon {PREDICTION_LENGTH}, "
      f"batch {SERIES_BATCH_SIZE}, {N_ASSETS} assets, rounds 1-12")
print(f"  rounds done  : {len(round_seconds)}/{len(list(ROUNDS))} "
      f"({sum(round_seconds.values()) / 60:.1f} min of inference)")
print()
for num_samples in SAMPLE_COUNTS:
    done = [rec for rec in records if rec["num_samples"] == num_samples]
    status = "COMPLETE" if len(done) == len(list(ROUNDS)) else "INCOMPLETE"
    print(f"  {num_samples:>4} samples: {len(done):>2}/12 rounds {status:<10} "
          f"-> {config_dir(num_samples)}")
if failures:
    print(f"\n  FAILED rounds: {[r for r, _ in failures]}")
print(f"\n  all files verified: {ALL_VERIFIED} | nested prefix check: {PREFIX_OK}")
print("  post-processing   : none (no four-week returns, quintiles, RPS, DRE change)")
print("\nNext: copy the completed samples_*/ folders back into TSFM Inference and Sampling/scenario_outputs/ "
      "and score them with scripts/evaluate_m6_rps.py.")